In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys
import seaborn as sns
import scipy.stats as st
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

In [ ]:
folder = Path('../../../data/this_project/1_e_coli_batch_cultures/')
div_folder = Path('../../../data/this_project/5_div/')

## Map metabolomics names to ECMDB IDs

In [ ]:
name_to_bigg = pd.read_csv(div_folder / '5A_metabolomics_name_to_bigg.csv')
meta_info = pd.read_csv(div_folder / '5C_metabolite_info.csv', index_col=0)

merged = name_to_bigg.merge(
    meta_info[['Metabolite', 'ECMDB ID', 'ECMDB name']],
    left_on='Metabolite', right_on='Metabolite',
    how='left'
)

no_ecmdb = merged[merged['ECMDB ID'].isna()][['Metabolite', 'BiGG ID']]
print(f"Mapped: {merged['ECMDB ID'].notna().sum()}/{len(merged)}")
print(f"\nNo ECMDB ID ({len(no_ecmdb)} metabolites):")
print(no_ecmdb.to_string(index=False))

Mapped: 41/128

No ECMDB ID (87 metabolites):
                            Metabolite   BiGG ID
                    1,3-diaminopropane   13dampp
                 2,5-dihydroxybenzoate     23dhb
                    2-aminoisobutyrate       NaN
                     2-hydroxybutyrate       ghb
                2-hydroxyglutaric acid   S2hglut
           3-(4-hydroxyphenyl)pyruvate     34hpp
                 3-hydroxyanthranilate       NaN
                     3-hydroxybutyrate       ghb
                3-hydroxyglutaric acid   S2hglut
                3-hydroxyphenylacetate       NaN
                  4-acetamidobutanoate   4aabutn
                    4-imidazoleacetate     im4ac
                             Adenosine       and
      Adenosine 3',5'-cyclic phosphate    23camp
                               Adipate       NaN
                                 Aicar     aicar
                             Allantoin     alltn
                   Alpha-aminobutyrate       NaN
             Alpha-gluc

In [ ]:
import json

# Load ECMDB JSON and build a flat name -> met_id lookup
with open('../../../data/other/ecmdb.json') as f:
    ecmdb_data = json.load(f)

ecmdb_name_map = {}
for entry in ecmdb_data:
    mid = entry.get('met_id')
    if not mid:
        continue
    for field in ['name', 'moldb_iupac', 'moldb_traditional_iupac']:
        v = entry.get(field, '')
        if v:
            ecmdb_name_map[v.lower()] = mid
    if entry.get('wikipedia_link'):
        ecmdb_name_map[entry['wikipedia_link'].lower().replace('_', ' ')] = mid

# Alternative names for metabolites not matched by direct name
alt_name_lookup = {
    '2-aminoisobutyrate':              'ECMDB21403',  # 2-amino-2-methylpropanoic acid
    '2-hydroxybutyrate':               'ECMDB24007',  # 2-hydroxybutanoic acid
    '3-(4-hydroxyphenyl)pyruvate':     'ECMDB00707',  # 4-hydroxyphenylpyruvic acid
    '3-hydroxyanthranilate':           'ECMDB01476',  # 3-hydroxyanthranilic acid
    '3-hydroxybutyrate':               'ECMDB21418',  # 3-hydroxybutanoic acid
    'adenosine 3\',5\'-cyclic phosphate': 'ECMDB00058',  # cyclic adenosine monophosphate
    'alpha-glucose (cl adduct)':       'ECMDB00122',  # D-Glucose
    'anthranilate':                    'ECMDB01123',  # anthranilic acid
    'deoxycarnitine (deoxy-c0)':       'ECMDB21336',  # gamma-butyrobetaine
    'diaminopimelate':                 'ECMDB21665',  # diaminopimelic acid
    'dihydroorotate':                  'ECMDB00528',  # dihydroorotic acid
    'guanidinosuccinate/n-acetylaspartate': 'ECMDB21401',  # N-acetylaspartate
    'hexoses including glucose and fructose': 'ECMDB00660',  # D-Fructose
    'n-acetylglutamate':               'ECMDB21429',  # N-acetylglutamic acid
    'n-alpha-acetyllysine':            'ECMDB24023',  # N6-acetyllysine
    'nicotinate':                      'ECMDB01488',  # nicotinic acid
    'o-succinyl-homoserine':           'ECMDB01418',  # O-succinyl-L-homoserine
    'pantothenate':                    'ECMDB00210',  # pantothenic acid
    'salicylate':                      'ECMDB21437',  # salicylic acid
    'uridine diphosphate hexose':      'ECMDB04171',  # UDP-glucose
    'uridine monophosphate':           'ECMDB00288',  # uridine 5'-monophosphate
    'urocanate':                       'ECMDB21396',  # urocanic acid
    'camp':                            'ECMDB00058',  # cyclic AMP
    'ureidopropionate':                'ECMDB00026',  # ureidopropionic acid
}

def lookup_ecmdb(metabolite_name):
    key = metabolite_name.lower().strip()
    if key in ecmdb_name_map:
        return ecmdb_name_map[key]
    if key in alt_name_lookup:
        return alt_name_lookup[key]
    return None

# Apply to no_ecmdb
no_ecmdb_mapped = no_ecmdb.copy()
no_ecmdb_mapped['ECMDB ID'] = no_ecmdb_mapped['Metabolite'].apply(lookup_ecmdb)

# Look up ECMDB name for matched entries
ecmdb_id_to_name = {d['met_id']: d['name'] for d in ecmdb_data if d.get('met_id')}
no_ecmdb_mapped['ECMDB name'] = no_ecmdb_mapped['ECMDB ID'].map(ecmdb_id_to_name)

newly_mapped = no_ecmdb_mapped[no_ecmdb_mapped['ECMDB ID'].notna()]
still_unmapped = no_ecmdb_mapped[no_ecmdb_mapped['ECMDB ID'].isna()]

print(f"Additionally mapped via ecmdb.json: {len(newly_mapped)}")
print(f"Still no ECMDB ID: {len(still_unmapped)}")
print()
print("Newly mapped:")
print(newly_mapped[['Metabolite', 'BiGG ID', 'ECMDB ID', 'ECMDB name']].to_string(index=False))


Additionally mapped via ecmdb.json: 58
Still no ECMDB ID: 29

Newly mapped:
                            Metabolite BiGG ID   ECMDB ID                    ECMDB name
                    2-aminoisobutyrate     NaN ECMDB21403        2-Aminoisobutyric acid
                     2-hydroxybutyrate     ghb ECMDB24007         2-Hydroxybutyric acid
                2-hydroxyglutaric acid S2hglut ECMDB00606      D-2-Hydroxyglutaric acid
           3-(4-hydroxyphenyl)pyruvate   34hpp ECMDB00707   4-Hydroxyphenylpyruvic acid
                 3-hydroxyanthranilate     NaN ECMDB01476     3-Hydroxyanthranilic acid
                     3-hydroxybutyrate     ghb ECMDB21418         3-Hydroxybutyric acid
                             Adenosine     and ECMDB00050                     Adenosine
      Adenosine 3',5'-cyclic phosphate  23camp ECMDB00058                    Cyclic AMP
                                 Aicar   aicar ECMDB01517                         AICAR
                             Allantoin   all

In [ ]:
print(f"Metabolites with no ECMDB ID ({len(still_unmapped)}):")
print(still_unmapped[['Metabolite', 'BiGG ID']].to_string(index=False))

Metabolites with no ECMDB ID (29):
                   Metabolite   BiGG ID
           1,3-diaminopropane   13dampp
        2,5-dihydroxybenzoate     23dhb
       3-hydroxyglutaric acid   S2hglut
       3-hydroxyphenylacetate       NaN
         4-acetamidobutanoate   4aabutn
           4-imidazoleacetate     im4ac
                      Adipate       NaN
          Alpha-aminobutyrate       NaN
             Cdp-ethanolamine     cdpea
                     Creatine       crn
                   Creatinine      crtn
    Glutarylcarnitine (c5-dc)       NaN
       Hexanoylcarnitine (c6)       NaN
                      Hexoses       NaN
         Hydroxyphenyllactate 2hyoxplac
 Isovalerylcarnitine (c5/ic5)       NaN
                   Kynurenate       NaN
     Malonylcarnitine (c3:dc)       NaN
                 Metanephrine      mepi
              Methylglutarate   3mglutr
              Methylguanidine       NaN
              N-acetylleucine  acleu__L
        N-acetylphenylalanine       NaN
     

In [ ]:
# Build full annotation DataFrame for all 128 metabolites
# Start from merged (has ECMDB ID/name from 5C_metabolite_info where available)
# Fill gaps with ecmdb.json matches, then bring in extra columns from meta_info

extra_cols = ['Metabolite', 'Metabolite id']

df_all = merged.copy()

# Fill ECMDB ID / name for rows that were found via ecmdb.json
ecmdb_fill = no_ecmdb_mapped.set_index('Metabolite')[['ECMDB ID', 'ECMDB name']]
df_all = df_all.set_index('Metabolite')
df_all.update(ecmdb_fill)
df_all = df_all.reset_index()

# Merge in extra annotations from meta_info
df_all = df_all.merge(
    meta_info[extra_cols].rename(columns={'Metabolite id': 'BiGG ID (meta_info)'}),
    on='Metabolite', how='left'
)

# Use BiGG ID from 5A where available, fall back to meta_info
df_all['BiGG ID'] = df_all['BiGG ID'].fillna(df_all['BiGG ID (meta_info)'])
df_all = df_all.drop(columns=['BiGG ID (meta_info)'])

df_all = df_all[['Metabolite', 'BiGG ID', 'ECMDB ID', 'ECMDB name']]

print(f"Total metabolites: {len(df_all)}")
print(f"With ECMDB ID:     {df_all['ECMDB ID'].notna().sum()}")
print(f"Without ECMDB ID:  {df_all['ECMDB ID'].isna().sum()}")
df_all.to_csv(div_folder / '5X_metabolite_annotation.csv', index=False)

Total metabolites: 128
With ECMDB ID:     99
Without ECMDB ID:  29


In [ ]:
from reframed import load_cbmodel

model = load_cbmodel('../../../models/e_coli/iML1515.xml')

# Extract unique BiGG base IDs from model metabolites (strip M_ prefix and _c/_e/_p suffix)
model_bigg_ids = set()
for met_id in model.metabolites:
    base = met_id
    if base.startswith('M_'):
        base = base[2:]
    # Strip compartment suffix (_c, _e, _p, etc.)
    if base.rsplit('_', 1)[-1] in ('c', 'e', 'p'):
        base = base.rsplit('_', 1)[0]
    model_bigg_ids.add(base)

def in_model(bigg_id):
    if pd.isna(bigg_id) or bigg_id == '':
        return None
    return bigg_id in model_bigg_ids

df_all['In iML1515'] = df_all['BiGG ID'].apply(in_model)

n_in  = df_all['In iML1515'].eq(True).sum()
n_out = df_all['In iML1515'].eq(False).sum()
n_na  = df_all['In iML1515'].isna().sum()
print(f"In model:          {n_in}")
print(f"Not in model:      {n_out}")
print(f"No BiGG ID:        {n_na}")
df_all
df_all.to_csv(div_folder / '5X_metabolite_annotation.csv', index=False)

In model:          85
Not in model:      17
No BiGG ID:        26
